# MCBench: Measuring Metacognitive Capability in AI via Signal Detection Theory

**Hackathon:** Measuring Progress Toward AGI – Cognitive Abilities  
**Author:** Ivo Georgiev  
**Dataset:** MMLU-Pro + GPQA Diamond (1,698 curated items; 600 eval / 900 held-out)

## Overview

MCBench measures **metacognitive capability** — not just whether a model gets answers right,
but whether it *knows what it knows*. This is a key AGI milestone: human-level metacognition
allows us to allocate cognitive effort, recognise uncertainty, and avoid confident mistakes.

### Three-run pipeline
| Run | Task | What it measures |
|-----|------|------------------|
| 1   | Zero-shot MCQ answer | Type 1 accuracy (d′) |
| 2   | Self-recognition from 3 responses | Attribution capability (SRS) |
| 3   | Two-turn confidence elicitation | Metacognitive sensitivity (meta-d′) |

### Metacognitive Capability Index (MCI)
$$\text{MCI} = 0.65 \cdot \max(0, M\text{-ratio}) + 0.35 \cdot \text{SRS} - \gamma \cdot \text{ECE}$$

where $M\text{-ratio} = \text{meta-}d' / d'$ (Type 2 AUROC method), SRS = Self-Recognition Score,
ECE = Expected Calibration Error, and $\gamma$ is dynamic (penalises constant-confidence outputs).

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
import sys, os
from pathlib import Path

# Add repo root to path when running on Kaggle
REPO_ROOT = Path('/kaggle/input/mcbench-metacognitive-benchmark')
if REPO_ROOT.exists():
    sys.path.insert(0, str(REPO_ROOT))
else:
    REPO_ROOT = Path('.').resolve()
    sys.path.insert(0, str(REPO_ROOT))

print(f'Working from: {REPO_ROOT}')

In [ ]:
# ── Register kbench tasks ─────────────────────────────────────────────────────
# Importing these modules registers the @kbench.task decorators
import kaggle_benchmarks as kbench

from src.benchmark.run1_zeroshot import run1_zeroshot
from src.benchmark.run2_recognition import run2_recognition
from src.benchmark.run3_metacognition import run3_metacognition

print('Registered tasks:', [run1_zeroshot.name, run2_recognition.name, run3_metacognition.name])

In [ ]:
# ── Select the primary benchmark task for the Kaggle leaderboard ──────────────
# Run 3 (metacognitive confidence) is the observable output;
# MCI aggregation happens in post-processing below.
%choose run3_metacognition

In [ ]:
# ── Load models from Kaggle model proxy ──────────────────────────────────────
from kaggle_benchmarks.kaggle import load_available_models, is_configured

if is_configured():
    models_dict = load_available_models()
    models = list(models_dict.values())
    print(f'Loaded {len(models)} models: {[m.name for m in models]}')
else:
    print('Model proxy not configured — running in demo mode with mock models')
    models = None  # pipeline will raise a helpful error

print(f'Judge model: {kbench.judge_llm}')

In [ ]:
# ── Run the full MCBench pipeline ─────────────────────────────────────────────
from src.pipeline import run_mcbench

leaderboard = run_mcbench(
    models=models,
    # Uncomment for a quick smoke test:
    # max_items=50,
)

In [ ]:
# ── Display leaderboard ───────────────────────────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

display_cols = ['model', 'mci', 'm_ratio', 'd_prime', 'type2_auroc', 'srs', 'ece', 'reliable']
print('\n=== MCBench Leaderboard ===')
display(leaderboard[display_cols].style
    .format({'mci': '{:.4f}', 'm_ratio': '{:.4f}', 'd_prime': '{:.4f}',
             'type2_auroc': '{:.4f}', 'srs': '{:.4f}', 'ece': '{:.4f}'})
    .background_gradient(subset=['mci'], cmap='YlOrRd')
    .set_caption('MCBench Rankings — higher MCI = stronger metacognitive capability')
)

In [ ]:
# ── Visualise results ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('MCBench Results: Metacognitive Capability Decomposition', fontsize=13)

# Sort for display
lb = leaderboard.sort_values('mci', ascending=True)

# MCI score
axes[0].barh(lb['model'], lb['mci'], color='steelblue')
axes[0].set_xlabel('MCI (Metacognitive Capability Index)')
axes[0].set_title('Overall MCI Score')
axes[0].axvline(0, color='red', linestyle='--', alpha=0.5)

# M-ratio (metacognitive efficiency)
axes[1].barh(lb['model'], lb['m_ratio'].clip(lower=0), color='darkorange')
axes[1].set_xlabel('M-ratio (meta-d\'/d\'), clipped at 0')
axes[1].set_title('Metacognitive Efficiency')
axes[1].axvline(1, color='green', linestyle='--', alpha=0.5, label='optimal=1')
axes[1].legend(fontsize=8)

# Component breakdown: SRS vs ECE
x = range(len(lb))
axes[2].scatter(lb['srs'], lb['ece'], s=80, color='purple')
for _, row in lb.iterrows():
    axes[2].annotate(row['model'], (row['srs'], row['ece']), 
                     fontsize=7, xytext=(3, 3), textcoords='offset points')
axes[2].set_xlabel('SRS (Self-Recognition Score, higher=better)')
axes[2].set_ylabel('ECE (Calibration Error, lower=better)')
axes[2].set_title('Attribution vs Calibration')

plt.tight_layout()
plt.savefig('mcbench_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: mcbench_results.png')

In [ ]:
# ── Save results ──────────────────────────────────────────────────────────────
leaderboard.to_csv('mcbench_leaderboard.csv', index=True)
print('Leaderboard saved: mcbench_leaderboard.csv')
print(f'\nTop model: {leaderboard.iloc[0]["model"]} (MCI={leaderboard.iloc[0]["mci"]:.4f})')
print(f'Bottom model: {leaderboard.iloc[-1]["model"]} (MCI={leaderboard.iloc[-1]["mci"]:.4f})')